# T6 — Build the cross-modal image index (CLIP, GPU)

The **embed-the-image** arm that H3 puts on trial (PROPOSAL §3.2, §5.3). A CLIP encoder maps each
figure/slide image **and** a text query into one shared, L2-normalized space, so retrieval is
genuinely cross-modal (`s_V(q,c) = ⟨ψ(q), ψ(image_c)⟩`). Encoding hundreds of images with a vision
transformer is GPU-bound — hence Colab — while the local box only runs the light query side.

**Workflow**
1. Locally: `python scripts/build_image_index.py --pack` → `data/processed/image_payload.zip`
   (chunks.json + only the referenced slide PNGs).
2. Set runtime to **GPU** (Runtime → Change runtime type → T4), then run all cells and upload that zip.
3. Download `image_index.zip`, unzip into `data/processed/` so you get `data/processed/image_index/`.
4. Locally: `python scripts/probe_image_search.py "..."` to sanity-check hits (T6 DoD).

The artifact format matches `mmrag.image_embeddings.ImageIndex.save`:
`image_vecs.npy` (N×D float32, normalized) + `image_index.json` (`{model, dim, chunks}`).

In [ ]:
# 1. Deps + GPU check. Colab ships torch+CUDA; we just need sentence-transformers (CLIP).
!pip -q install "sentence-transformers>=2.7" pillow
import torch
print("CUDA available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

In [ ]:
# 2. Upload data/processed/image_payload.zip (produced by build_image_index.py --pack) and unzip.
import json, zipfile, pathlib
from google.colab import files

uploaded = files.upload()  # pick image_payload.zip
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as z:
    z.extractall("payload")

manifest = json.loads(pathlib.Path("payload/manifest.json").read_text(encoding="utf-8"))
print(f"{len(manifest)} visual chunks across {len({c['source'] for c in manifest})} decks")

In [ ]:
# 3. Encode every image with CLIP on the GPU (one shared text-image space, normalized).
import numpy as np
from PIL import Image
from sentence_transformers import SentenceTransformer

MODEL = "clip-ViT-B-32"  # keep in sync with config.yaml image_embeddings.model
model = SentenceTransformer(MODEL, device="cuda" if torch.cuda.is_available() else "cpu")

paths = [f"payload/{c['arcname']}" for c in manifest]
images = [Image.open(p).convert("RGB") for p in paths]
vecs = model.encode(images, batch_size=64, convert_to_numpy=True,
                    normalize_embeddings=True, show_progress_bar=True).astype("float32")
for im in images:
    im.close()
print("vectors:", vecs.shape)

In [ ]:
# 4. Cross-modal probe — confirm a text query surfaces sane slides (T6 DoD, on GPU).
probes = [
    "scaled dot-product attention architecture diagram",
    "a scatter plot of data points",
    "table comparing model results",
]
for q in probes:
    qv = model.encode([q], convert_to_numpy=True, normalize_embeddings=True)[0].astype("float32")
    scores = vecs @ qv
    top = np.argsort(scores)[::-1][:5]
    print("Q:", q)
    for i in top:
        c = manifest[i]
        cap = (c.get("figure_caption") or "").replace("\n", " ")[:80]
        print(f"  {scores[i]:.3f}  {c['source']} p{c['page']}  {cap}")
    print()

In [ ]:
# 5. Save in the ImageIndex.save format and zip for download.
#    Drop the 'arcname' helper; keep each chunk's original image_path so the LOCAL
#    machine (which has the PNGs) can open them for probing / fusion.
import os
out = pathlib.Path("image_index"); out.mkdir(exist_ok=True)
np.save(out / "image_vecs.npy", vecs)
chunks = [{k: v for k, v in c.items() if k != "arcname"} for c in manifest]
meta = {"model": MODEL, "dim": int(vecs.shape[1]), "chunks": chunks}
(out / "image_index.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")

with zipfile.ZipFile("image_index.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in out.iterdir():
        z.write(f, f"image_index/{f.name}")
files.download("image_index.zip")
print("Done. Unzip image_index.zip into data/processed/ on your machine.")